In [22]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import torch
from data import Batch
from losses import CorrelationRegressionLoss, ListwiseRankingLoss, MultiTaskLoss, ReturnLoss
from model import ContextEnrichedEmbedder, ModelConfig
from targets import cross_sectional_demean, residualize_returns
import itertools
import torch
from data import Batch
from losses import CorrelationRegressionLoss, ListwiseRankingLoss, MultiTaskLoss, ReturnLoss
from model import ContextEnrichedEmbedder, ModelConfig
from targets import cross_sectional_demean, residualize_returns

CORR_LOSSES = {"regression": CorrelationRegressionLoss, "ranking": ListwiseRankingLoss}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
company = pd.read_csv("../../data/03_gold/company.csv")
makro = pd.read_csv("../../data/03_gold/makro.csv")
zins = pd.read_csv("../../data/03_gold/zins.csv")
currency = pd.read_csv("../../data/03_gold/currency.csv")
commodity = pd.read_csv("../../data/03_gold/commodity.csv")

In [3]:
all_countries = sorted(set(zins["iso3"]) | set(makro["iso3"]) | set(currency["iso3"]))
country_to_id = {c: i + 1 for i, c in enumerate(all_countries)}
zins_vars = sorted(zins["item_description"].unique())       # 3 Stück
makro_vars = sorted(makro["item_description"].unique())
currency_vars = sorted(currency["item_description"].unique())  
commodity_vars = sorted(commodity["commodity_name"].unique())  

entity_to_id = {}
idx = 1
for v in zins_vars:
    entity_to_id[("zins", v)] = idx; idx += 1
for v in makro_vars:
    entity_to_id[("makro", v)] = idx; idx += 1
for v in currency_vars:
    entity_to_id[("currency", v)] = idx; idx += 1
for v in commodity_vars:
    entity_to_id[("commodity", v)] = idx; idx += 1

In [11]:
# 2. Feste Knotenliste definieren (bestimmt die exakte Reihenfolge in ctx_x!)
node_specs = []  # Liste von (df_name, country, indicator)
types, ents, ctrs = [], [], []

# Typ 0: Zins
for c in all_countries:
    for v in zins_vars:
        node_specs.append(("zins", c, v))
        types.append(0)
        ents.append(entity_to_id[("zins", v)])
        ctrs.append(country_to_id[c])

# Typ 1: Makros
for c in all_countries:
    for v in makro_vars:
        node_specs.append(("makro", c, v))
        types.append(1)
        ents.append(entity_to_id[("makro", v)])
        ctrs.append(country_to_id[c])

# Typ 2: Currency (alle Länder außer USA)
for c in all_countries: 
    if c != "USA":
        for v in currency_vars:
            node_specs.append(("currency", c, v))
            types.append(2)
            ents.append(entity_to_id[("currency", v)])
            ctrs.append(country_to_id[c])
# Typ 3: Commodity (Keine Länderzuordnung, deswegen ctrs = 0)
for v in commodity_vars:
    node_specs.append(("commodity", 0, v))
    types.append(3)
    ents.append(entity_to_id[("commodity", v)])
    ctrs.append(0)

node_type    = torch.tensor(types, dtype=torch.long)
node_entity  = torch.tensor(ents, dtype=torch.long)
node_country = torch.tensor(ctrs, dtype=torch.long)

nodes = (node_type, node_entity, node_country)

In [5]:
commodity['iso3'] = 0
commodity = commodity.rename(columns={'ret_1m': 'ret_1y', 'ret_3m': 'ret_2y', 'ret_6m': 'ret_3y', 'ret_12m': 'ret_4y', 'ret_24m': 'ret_5y', 'vol_3m': 'vol_2y', 'vol_6m': 'vol_3y', 'vol_12m': 'vol_4y', 'vol_24m': 'vol_5y', 'zscore_24m': 'zscore_5y', 'commodity_name': 'item_description' })

In [20]:
def build_monthly_batch(
    date_t,
    df_company_t: pd.DataFrame,   # Zeilen des Monats t, sortiert nach EntityCode (B_t Zeilen)
    firm_feat_cols: list[str],    # die 74 Feature-Spalten
    ctx_lookup: dict,             # (src, country, indicator, date) -> np.ndarray(10,)
    daily_returns_fwd: pd.DataFrame | None = None, # Index=Handelstage t+1..t+3, Columns=EntityCodes
) -> Batch:
    # 1. Firmen-Features: (B_t, 74)
    firm_x = torch.tensor(df_company_t[firm_feat_cols].to_numpy(), dtype=torch.float32)
    B_t = firm_x.shape[0]

    # 2. Kontext-Features & Maske: (1, N, 10) und (1, N)
    N = len(node_specs)
    ctx_arr = torch.zeros((1, N, 10), dtype=torch.float32)
    ctx_mask = torch.zeros((1, N), dtype=torch.bool)

    for j, (src, c, ind) in enumerate(node_specs):
        vec = ctx_lookup.get((src, c, ind, date_t))
        if vec is not None and not pd.isna(vec).all():
            ctx_arr[0, j] = torch.tensor(vec, dtype=torch.float32)
            ctx_mask[0, j] = True

    # 3. Targets (für Train/Val; im Out-of-Sample-Test einfach None lassen)
    fwd_resid = None
    fwd_ret = None
    if daily_returns_fwd is not None:
        # Spalten exakt auf die Firmenreihenfolge in df_company_t ausrichten!
        firms = df_company_t["EntityCode"].tolist()
        R = torch.tensor(daily_returns_fwd[firms].fillna(0.0).to_numpy(), dtype=torch.float32)
        fwd_resid = residualize_returns(R, "ew_market")
        fwd_ret = cross_sectional_demean(R.sum(dim=0))

    return Batch(
        firm_x=firm_x,
        ctx_x=ctx_arr,
        ctx_mask=ctx_mask,
        firm_to_ctx=torch.zeros(B_t, dtype=torch.long),
        fwd_resid=fwd_resid,
        fwd_ret=fwd_ret,
    )


In [15]:
# Die 10 einheitlichen Feature-Spalten deiner Kontext-DataFrames
ctx_feat_cols = [
    "ret_1y", "ret_2y", "ret_3y", "ret_4y", "ret_5y",
    "vol_2y", "vol_3y", "vol_4y", "vol_5y", "zscore_5y",
]

ctx_lookup = {}

def fill_lookup(df: pd.DataFrame, src_name: str, country_col: str | None, ind_col: str, date_col: str = "date"):
    df = df.copy()
    # Datum auf Monatsende normieren, damit es exakt zu date_t passt
    df[date_col] = pd.to_datetime(df[date_col]) + pd.offsets.MonthEnd(0)
    
    vals = df[ctx_feat_cols].to_numpy(dtype=np.float32)
    countries = df[country_col].tolist() if country_col is not None else [0] * len(df)
    indicators = df[ind_col].tolist()
    dates = df[date_col].tolist()

    for c, ind, d, vec in zip(countries, indicators, dates, vals):
        ctx_lookup[(src_name, c, ind, d)] = vec

# Jetzt deine 4 DataFrames in das Nachschlagebuch füllen:
# (Passe die Spaltennamen für Land/Indikator/Datum ggf. an deine DataFrames an!)
fill_lookup(zins,      src_name="zins",      country_col="iso3", ind_col="item_description",    date_col="date")
fill_lookup(makro,     src_name="makro",     country_col="iso3", ind_col="item_description",    date_col="date")
fill_lookup(currency,  src_name="currency",  country_col="iso3", ind_col="item_description",    date_col="date")
fill_lookup(commodity, src_name="commodity", country_col=None,   ind_col="item_description",    date_col="date")

In [8]:
weekly_returns = pd.read_csv("../../data/03_gold/weekly_returns.csv")
# 1. Datumsformate in company & weekly_returns sicherstellen
company["date"] = pd.to_datetime(company["date"]) + pd.offsets.MonthEnd(0)
weekly_returns.index = pd.to_datetime(weekly_returns['Date'])

# Alle 74 Feature-Spalten des Company-DataFrames (alles außer EntityCode und Date)
firm_feat_cols = [c for c in company.columns if c not in ["EntityCode", "date"]]

def create_batches(dates: list[pd.Timestamp], is_test: bool = False, horizon_months: int = 3) -> list[Batch]:
    batches = []
    for date_t in dates:
        # Firmen des Monats t ohne fehlende Werte in den 74 Features
        df_comp_t = company[company["date"] == date_t].dropna(subset=firm_feat_cols).copy()
        if df_comp_t.empty:
            continue

        if is_test:
            # Für den Test-Stichtag brauchen wir keine Targets (daily_returns_fwd=None)
            df_comp_t = df_comp_t.sort_values("EntityCode")
            b = build_monthly_batch(
                date_t=date_t,
                df_company_t=df_comp_t,
                firm_feat_cols=firm_feat_cols,
                ctx_lookup=ctx_lookup,
                daily_returns_fwd=None,
            )
            batches.append(b)
        else:
            # Für Train & Val: Zukünftige wöchentliche Renditen über die nächsten 3 Monate ausschneiden
            fwd_start = date_t + pd.Timedelta(days=1)
            fwd_end = (date_t + pd.DateOffset(months=horizon_months)) + pd.offsets.MonthEnd(0)
            fwd_rets = weekly_returns.loc[fwd_start:fwd_end]

            # Nur Firmen behalten, für die wir im Ziel-Fenster auch Renditen haben (min. 80% der Wochen)
            valid_firms = [
                f for f in df_comp_t["EntityCode"]
                if f in fwd_rets.columns and fwd_rets[f].notna().mean() >= 0.8
            ]
            df_comp_t = df_comp_t[df_comp_t["EntityCode"].isin(valid_firms)].sort_values("EntityCode")

            b = build_monthly_batch(
                date_t=date_t,
                df_company_t=df_comp_t,
                firm_feat_cols=firm_feat_cols,
                ctx_lookup=ctx_lookup,
                daily_returns_fwd=fwd_rets[valid_firms],
            )
            batches.append(b)

    return batches

In [9]:
all_months = sorted(company["date"].unique())

# Beispiel für Cutoff 31.12.2023 (bei 3-Monats-Target endet der letzte Val-Target am 31.12.2023):
train_dates = [d for d in all_months if pd.Timestamp("2021-06-30") <= d <= pd.Timestamp("2023-06-30")]
val_dates   = [d for d in all_months if pd.Timestamp("2023-07-31") <= d <= pd.Timestamp("2023-09-30")]
test_dates  = [pd.Timestamp("2023-12-31")]

train_batches = create_batches(train_dates, is_test=False, horizon_months=3)
val_batches   = create_batches(val_dates,   is_test=False, horizon_months=3)
test_batches  = create_batches(test_dates,  is_test=True)

print(f"Fertig! Train-Batches: {len(train_batches)}, Val-Batches: {len(val_batches)}, Test-Batches: {len(test_batches)}")

Fertig! Train-Batches: 25, Val-Batches: 3, Test-Batches: 0


In [34]:
import copy
import torch
from model import ContextEnrichedEmbedder, ModelConfig
from losses import CorrelationRegressionLoss, ListwiseRankingLoss, MultiTaskLoss, ReturnLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Konfiguration festlegen
cfg = ModelConfig(
    firm_in_dim=74,                      # 64 VAE + 10 Return/Vola-Features
    ctx_in_dim=10,                       # deine 10 Zeitreihen-Features pro Kontextknoten
    n_node_types=4,                      # 0=Zins, 1=Makros, 2=Currency, 3=Commodity
    n_entities=len(entity_to_id) + 1,    # +1 wegen padding_idx=0[cite: 1]
    n_countries=len(country_to_id) + 1,  # +1 wegen padding_idx=0[cite: 1]
    d_model=32,                          # interne Arbeitsdimension[cite: 1]
    emb_dim=16,                          # finale Embedding-Dimension für k-Means[cite: 1]
    attn="gatv2",                        # "dot", "gatv2", "gatv1", "film" oder "meanpool"[cite: 1]
    n_heads=4,
    n_layers=1,
    topk=15,                             # optional: fokussiert auf die 15 stärksten Kontextknoten[cite: 1]
    dropout=0.2,
    ctx_self_layers=0,                   # 0 = kein Austausch zwischen Kontextknoten untereinander[cite: 1]
    ret_head=True,                       # Hilfsziel für zukünftige Rendite aktivieren[cite: 1]
)

# 2. Modell und Multi-Task-Loss auf GPU/CPU schieben
model = ContextEnrichedEmbedder(cfg, *nodes).to(device)

# Hauptziel: Korrelationsmatrix-Loss + Hilfsziel: Rendite-Loss (ret_weight=0.1)
criterion = MultiTaskLoss(
    CorrelationRegressionLoss(fisher=False, learn_affine=False),         # alternativ: ListwiseRankingLoss()
    ReturnLoss(),
    ret_weight=0.1
).to(device)

# 3. Optimizer (wichtig: criterion.parameters() mit aufnehmen, falls der Loss lernbare Gewichte hat!)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=1e-3,
    weight_decay=1e-2,
)

In [35]:
import copy
import torch

# Hilfsfunktion: Zieht für einen Trainingsschritt zufällig max_firms Firmen aus dem Monats-Batch
def subsample_batch(batch: Batch, max_firms: int = 2000) -> Batch:
    B = batch.firm_x.size(0)
    if B <= max_firms:
        return batch
    idx = torch.randperm(B, device=batch.firm_x.device)[:max_firms]
    return Batch(
        firm_x=batch.firm_x[idx],
        ctx_x=batch.ctx_x,
        ctx_mask=batch.ctx_mask,
        firm_to_ctx=batch.firm_to_ctx[idx],
        fwd_resid=batch.fwd_resid[:, idx] if batch.fwd_resid is not None else None,
        fwd_ret=batch.fwd_ret[idx] if batch.fwd_ret is not None else None,
    )

# Vor Start einmal den überlaufenden VRAM-Cache leeren
torch.cuda.empty_cache()

epochs = 100
patience = 15
max_firms_per_step = 2000  # Bei Bedarf auf 1500 senken, falls VRAM immer noch > 7 GB
best_val_loss = float("inf")
best_weights = None
no_improve = 0

history = {"train_loss": [], "val_loss": []}

for epoch in range(1, epochs + 1):
    # --- A) TRAINING ---
    model.train()
    train_loss_sum = 0.0
    
    perm = torch.randperm(len(train_batches)).tolist()
    for idx in perm:
        # Erst auf CPU subsamplen, dann nur den schlanken Batch auf die GPU schieben!
        batch = subsample_batch(train_batches[idx], max_firms_per_step).to(device)
        
        optimizer.zero_grad(set_to_none=True)
        out = model(batch)
        loss, logs = criterion(out, batch)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss_sum += loss.item()

    avg_train_loss = train_loss_sum / len(train_batches)

    # --- B) VALIDIERUNG (alle Firmen, aber im speichersparenden bfloat16 + no_grad) ---
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for val_batch in val_batches:
            b = val_batch.to(device)
            out = model(b)
            v_loss, _ = criterion(out, b)
            val_loss_sum += v_loss.item()

    avg_val_loss = val_loss_sum / len(val_batches)
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    # --- C) EARLY STOPPING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_weights = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early Stopping in Epoche {epoch}! Bester Val Loss: {best_val_loss:.5f}")
            break

# Beste Modellgewichte wiederherstellen
if best_weights is not None:
    model.load_state_dict(best_weights)

Epoch 001 | Train Loss: 1472.87172 | Val Loss: 474.62328
Epoch 005 | Train Loss: 1512.11340 | Val Loss: 474.21541
Epoch 010 | Train Loss: 1665.46060 | Val Loss: 473.55840
Epoch 015 | Train Loss: 1470.47770 | Val Loss: 473.01531
Epoch 020 | Train Loss: 1470.53900 | Val Loss: 472.99987
Epoch 025 | Train Loss: 1505.32512 | Val Loss: 474.32436
Epoch 030 | Train Loss: 1547.54314 | Val Loss: 473.50228
Epoch 035 | Train Loss: 1616.51557 | Val Loss: 471.75710
Epoch 040 | Train Loss: 1538.76660 | Val Loss: 473.76330
Epoch 045 | Train Loss: 1493.19331 | Val Loss: 474.14061
Epoch 050 | Train Loss: 1489.92659 | Val Loss: 473.59897
Epoch 055 | Train Loss: 1444.41640 | Val Loss: 472.79821
Epoch 060 | Train Loss: 1512.41786 | Val Loss: 467.59047
Epoch 065 | Train Loss: 1498.53163 | Val Loss: 473.92906
Epoch 070 | Train Loss: 1414.11901 | Val Loss: 474.20289
Epoch 075 | Train Loss: 1406.28414 | Val Loss: 472.67477
Epoch 080 | Train Loss: 1503.79838 | Val Loss: 458.82314
Epoch 085 | Train Loss: 1528.39

In [36]:
b = subsample_batch(train_batches[0], 2000).to(device)
out = model(b)
loss, logs = criterion(out, b)
print("Logs:", logs)
print("fwd_ret min/max/std:", b.fwd_ret.min().item(), b.fwd_ret.max().item(), b.fwd_ret.std().item())
print("fwd_resid min/max:", b.fwd_resid.min().item(), b.fwd_resid.max().item())

Logs: {'corr': 0.21561095118522644, 'ret': 0.3853280246257782, 'total': 0.25414374470710754}
fwd_ret min/max/std: -0.31995996832847595 0.93133944272995 0.19213137030601501
fwd_resid min/max: -0.2814517617225647 0.6848450303077698


In [31]:
model.eval()
with torch.no_grad():
    test_b = test_batches[0].to(device)
    out_test = model(test_b)

    # 1. Angereicherte, L2-normierte Firmen-Embeddings -> Input für k-Means Clustering[cite: 1, 3]
    z_test = out_test["z"].cpu().numpy()          # Shape: (B_test, 32)

    # 2. Attention-Gewichte -> Input für deine XAI-Auswertung (welches Unternehmen hört auf welchen Knoten)[cite: 1, 3]
    attn_test = out_test["attn"].cpu().numpy()    # Shape: (B_test, n_heads, N)

print("Fertige Embeddings Shape:", z_test.shape)
print("Attention-Matrix Shape:", attn_test.shape)

IndexError: list index out of range